In [ ]:
# Cell 1: Install dependencies
!pip install fastapi uvicorn pyngrok requests transformers underthesea -q
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q 2>/dev/null || pip install torch -q

In [ ]:
# Cell 2: Load files from Google Drive
from google.colab import drive
drive.mount('/drive')

import os, shutil

DRIVE_WORKER = '/drive/MyDrive/ABSA-Web/worker'
for f in ['app.py', 'crawler.py', 'inferencer.py', 'scoring.py']:
    shutil.copy(f'{DRIVE_WORKER}/{f}', f'/content/{f}')

os.makedirs('/content/phobert_model', exist_ok=True)
DRIVE_MODEL = '/drive/MyDrive/ABSA-Web/phobert_model'
for f in os.listdir(DRIVE_MODEL):
    shutil.copy(f'{DRIVE_MODEL}/{f}', f'/content/phobert_model/{f}')

print('Files loaded from Drive!')

In [ ]:
# Cell 3: Config — update these values
WORKER_API_KEY = 'worker-secret'        # must match backend WORKER_API_KEY env var
MAIN_API_URL   = 'https://absa-electronics-production.up.railway.app'
NGROK_TOKEN    = ''                     # get free token from ngrok.com dashboard

# ── Webshare proxies ──────────────────────────────────────────────────────────
# Paste all 10 proxy host:port here (same user/pass for all)
PROXY_USER = 'onwvqycb'
PROXY_PASS = 'b0461zzwd3ve'
PROXY_HOSTS = [
    '23.95.150.145:6114',
    '45.38.107.97:6014',
    # add the remaining 8 proxies here, one per line:
    # 'HOST:PORT',
]
SHOPEE_PROXY_LIST = ','.join(
    f'http://{PROXY_USER}:{PROXY_PASS}@{h}' for h in PROXY_HOSTS
)
# ─────────────────────────────────────────────────────────────────────────────

import os
os.environ['WORKER_API_KEY']    = WORKER_API_KEY
os.environ['SHOPEE_PROXY_LIST'] = SHOPEE_PROXY_LIST
print('Config set. Proxies:', len(PROXY_HOSTS))

In [ ]:
# Cell 4: Start FastAPI server in background thread
import threading
import uvicorn
from app import app

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)
print('Worker server started on port 8001')

In [ ]:
# Cell 5: Create ngrok tunnel + register with Main API
from pyngrok import ngrok, conf
import requests

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(8001)
worker_url = tunnel.public_url
print(f'Worker URL: {worker_url}')

# Register with Main API
r = requests.post(
    f'{MAIN_API_URL}/worker/register',
    json={'url': worker_url},
    headers={'x-worker-key': WORKER_API_KEY},
    timeout=10,
)
print(f'Registration: {r.status_code} {r.json()}')

In [ ]:
# Cell 6: Keep alive — run this cell to prevent Colab timeout
import time
print('Worker is running. Keep this cell alive to stay online.')
while True:
    time.sleep(60)
    print('... still alive')

In [ ]:
# Cell 7: Pre-crawl devices into DB (run ONCE after Cell 5)
# Crawls ~30 devices from Tiki and saves to Railway DB
import requests, time

QUERIES = [
    ('iphone 15', 3),
    ('iphone 14', 2),
    ('samsung galaxy s24', 3),
    ('samsung galaxy a55', 2),
    ('xiaomi 14t', 2),
    ('oppo reno 12', 2),
    ('vivo v30', 2),
    ('macbook air m2', 2),
    ('asus vivobook 15', 2),
    ('dell inspiron 15', 2),
    ('hp pavilion 15', 2),
    ('lenovo ideapad', 2),
]

worker_headers = {'x-worker-key': WORKER_API_KEY, 'Content-Type': 'application/json'}
total_saved = 0

for query, num in QUERIES:
    print(f'Crawling: {query} ({num} sản phẩm)...')
    try:
        r = requests.post(
            f'{worker_url}/crawl/search',
            json={'query': query, 'num_links': num, 'reviews_per_link': 30, 'platform': 'tiki'},
            headers=worker_headers,
            timeout=300,
        )
        if r.status_code == 200:
            devices = r.json().get('devices', [])
            save_r = requests.post(
                f'{MAIN_API_URL}/devices/batch',
                json={'devices': devices},
                headers={'x-worker-key': WORKER_API_KEY, 'Content-Type': 'application/json'},
                timeout=30,
            )
            saved = len(save_r.json().get('ids', [])) if save_r.status_code == 200 else 0
            total_saved += saved
            print(f'  OK: {len(devices)} crawled, {saved} saved to DB')
        else:
            print(f'  Error {r.status_code}: {r.text[:100]}')
    except Exception as e:
        print(f'  Exception: {e}')
    time.sleep(3)

print(f'\nDone! Tổng {total_saved} thiết bị đã lưu vào DB.')